In [0]:
%pip install xgboost -q
dbutils.library.restartPython()

In [0]:
import json
from pathlib import Path

import mlflow
import mlflow.xgboost
import pandas as pd
import xgboost as xgb

from mlflow import MlflowClient
from mlflow.models import infer_signature

from sklearn.datasets import load_iris
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

In [0]:
# ==========================================================
# 2. Configuración de MLflow
# ==========================================================

# Unity Catalog utiliza nombres de modelos con tres niveles:
# <catalog>.<schema>.<model>
REGISTERED_MODEL_NAME = "workspace.default.iris_random_forest"

# Nombre del Run que aparecerá en MLflow.
RUN_NAME = "xgboost-iris-challenger"

# Configuramos el Model Registry para trabajar con Unity Catalog.
mlflow.set_registry_uri("databricks-uc")


In [0]:
# ==========================================================
# 3. Carga del dataset
# ==========================================================

from pathlib import Path
import os

import pandas as pd

from sklearn.preprocessing import LabelEncoder


# ----------------------------------------------------------
# Detectar entorno Databricks
# ----------------------------------------------------------

try:
    spark
    IS_DATABRICKS = True
except NameError:
    IS_DATABRICKS = False


# ----------------------------------------------------------
# Función auxiliar para configuraciones
# ----------------------------------------------------------

def get_setting(
    setting_name: str,
    default_value: str,
    description: str,
) -> str:

    value = os.getenv(
        setting_name,
        default_value,
    )

    print(f"{description}: {value}")

    return value


# ----------------------------------------------------------
# Ruta del dataset
# ----------------------------------------------------------

DEFAULT_DATA_PATH = (
    "/Volumes/workspace/my_data/my_volumen/Iris.csv"
    if IS_DATABRICKS
    else "Iris.csv"
)

DATA_PATH = Path(
    get_setting(
        "IRIS_DATA_PATH",
        DEFAULT_DATA_PATH,
        "Ruta del dataset",
    )
)


# ----------------------------------------------------------
# Validación
# ----------------------------------------------------------

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset: {DATA_PATH}"
    )


# ----------------------------------------------------------
# Carga
# ----------------------------------------------------------

iris_df = pd.read_csv(DATA_PATH)

print("\nDataset cargado correctamente.")
print(f"Ruta: {DATA_PATH}")
print(f"Shape: {iris_df.shape}")

display(iris_df.head())


# ----------------------------------------------------------
# Definición de variables
# ----------------------------------------------------------

TARGET_COLUMN = "Species"

FEATURE_COLUMNS = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
]

X = iris_df[
    FEATURE_COLUMNS
].copy()


# ----------------------------------------------------------
# Codificación del target
# ----------------------------------------------------------

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    iris_df[TARGET_COLUMN]
)


# ----------------------------------------------------------
# Información final
# ----------------------------------------------------------

print("\nVariables predictoras:")
print(FEATURE_COLUMNS)

print("\nVariable objetivo:")
print(TARGET_COLUMN)

print("\nClases encontradas:")

for class_id, class_name in enumerate(
    label_encoder.classes_
):
    print(
        f"{class_id}: {class_name}"
    )

In [0]:
# ==========================================================
# 4. División Train/Test
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)